# Text-to-SQL Fine-tuning — Colab Training
**Model:** Qwen2.5-7B-Instruct | **Method:** QLoRA via Unsloth

Run the cells **top-to-bottom**. Read each section header before running.

> Requires Colab Pro (A100 / L4). T4 may OOM with Qwen2.5-7B even in 4-bit.

## Step 1 — Check GPU

In [1]:
import subprocess, torch

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('GPU             :', p.name)
    print('VRAM            : %.1f GB' % (p.total_memory / 1024**3))
else:
    raise RuntimeError('No GPU! Go to Runtime -> Change runtime type -> GPU')

Tue Sep 22 08:19:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2 — Install Dependencies

**Version notes:**
- `unsloth[colab-new]` from git: auto-detects CUDA/torch, installs matching xformers
- `trl < 0.10`: avoids `SFTTrainer` breaking API change in 0.10+
- `bitsandbytes >= 0.43`: required for `adamw_8bit` and 4-bit quantization
- Do **not** install `xformers` separately (unsloth handles it)
- Do **not** install `pyodbc` (Windows-only, not available on Colab Linux)

In [2]:
# Install unsloth first — it resolves compatible xformers/triton versions
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

# Install the rest of the training stack with safe version constraints
!pip install \
    "transformers>=4.50.3" \
    "trl>=0.8.0,<0.10.0" \
    "peft>=0.11.0" \
    "accelerate>=0.28.0" \
    "bitsandbytes>=0.43.0" \
    "datasets>=2.18.0" \
    "sqlglot>=23.0.0" \
    "pyyaml" \
    "numpy<2.0" \
    -q

print('All dependencies installed!')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 80.7 MB/s eta 0

In [3]:
!pip install --upgrade ipython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 639.0/639.0 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.2/86.2 kB 6.2 MB/s eta 0:00:00
  Attempting uninstall: traitlets
    Found existing installation: traitlets 5.7.1
    Uninstalling traitlets-5.7.1:
      Successfully uninstalled traitlets-5.7.1
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.5
    Uninstalling psutil-5.9.5:
      Successfully uninstalled psutil-5.9.5
  Attempting uninstall: ipython
    Found existing installation: ipython 7.34.0
    Uninstalling ipython-7.34.0:
      Successfully uninstalled ipython-7.34.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.9.6 requires trl!

In [4]:
# import importlib
# pkgs = ['unsloth','transformers','trl','peft','accelerate','bitsandbytes','datasets','sqlglot']
# for pkg in pkgs:
#     try:
#         m = importlib.import_module(pkg)
#         print('OK   %-20s %s' % (pkg, getattr(m, '__version__', '?')))
#     except ImportError:
#         print('MISS', pkg)

## Step 3 — Mount Google Drive

Checkpoints and the final model are saved to Drive so they survive session resets.

In [5]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ── CONFIGURE: change this path if you want a different folder on Drive ──
DRIVE_ROOT = '/content/drive/MyDrive/text-to-sql-finetune'
# ─────────────────────────────────────────────────────────────────────────

os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive root:', DRIVE_ROOT)

KeyboardInterrupt: 

## Step 4 — Setup Source Code

Choose **Option A** (clone from GitHub) or **Option B** (copy from Drive).

In [ ]:
import os, sys

# ── Option A: Clone from GitHub ────────────────────────────────────────
# from google.colab import userdata
# TOKEN = userdata.get('GITHUB_TOKEN')  # add via Colab Secrets (key icon)
# REPO  = 'YOUR_USERNAME/text-to-sql-finetune'
# !git clone https://{TOKEN}@github.com/{REPO}.git /content/text-to-sql-finetune

# ── Option B: Copy from Google Drive ────────────────────────────────────
!cp -r "{DRIVE_ROOT}/src"     /content/text-to-sql-finetune/
!cp -r "{DRIVE_ROOT}/configs" /content/text-to-sql-finetune/
!cp -r "{DRIVE_ROOT}/data"    /content/text-to-sql-finetune/
!cp -r "{DRIVE_ROOT}/prompts" /content/text-to-sql-finetune/
!cp -r "{DRIVE_ROOT}/requirements" /content/text-to-sql-finetune/

# ── Set PROJECT_DIR after the source is in place ────────────────────────
PROJECT_DIR = '/content/text-to-sql-finetune'  # adjust if using Option B

os.chdir(PROJECT_DIR)
# Add project root to sys.path so 'src.*' package imports work
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Install project as editable package (makes src.helpers, src.training, ... importable)
!pip install -e . -q

print('CWD      :', os.getcwd())
print('Contents :', os.listdir('.'))

## Step 5 — Config & Build Training Command

In [ ]:
def train(config):
    """Build a CLI argument string from a config dict."""
    args = ""
    for k, v in config.items():
        if k.startswith("_"):
            args += f'"{v}" '
        elif isinstance(v, str):
            args += f'--{k}="{v}" '
        elif isinstance(v, bool) and v:
            args += f"--{k} "
        elif isinstance(v, float) and not isinstance(v, bool):
            args += f"--{k}={v} "
        elif isinstance(v, int) and not isinstance(v, bool):
            args += f"--{k}={v} "
    return args

In [ ]:
cfg_file = '/content/text-to-sql-finetune/configs/training_config.yaml'

### (Optional) Override training config values

Edit the overrides below to point outputs at Drive and tune batch sizes.

In [ ]:
import yaml

with open(cfg_file, 'r') as f:
    config_data = yaml.safe_load(f)

# Redirect outputs to Drive so they survive session resets
config_data['training']['output_dir'] = DRIVE_ROOT + '/outputs/qwen2.5-7b-text2sql'

# save_checkpoints: save a LoRA checkpoint every N epochs.
# Set to None to disable. Use 1 on Colab for safest recovery.
config_data['training']['save_checkpoints'] = 10

# Reduce batch size to fit Colab VRAM (effective batch = 2 x 8 = 16)
config_data['training']['per_device_train_batch_size'] = 2
config_data['training']['gradient_accumulation_steps'] = 8

# Write the patched config back; train.py will read it via --config flag
with open(cfg_file, 'w') as f:
    yaml.dump(config_data, f, default_flow_style=False, allow_unicode=True)

print('Config updated:')
print("*" * 77)
print('  max_seq_length :', config_data['model']['max_seq_length'])
print('  name_or_path :', config_data['model']['name_or_path'])
print('  save_checkpoints :', config_data['training']['save_checkpoints'])
print('  batch (eff.)     :',
      config_data['training']['per_device_train_batch_size']
      * config_data['training']['gradient_accumulation_steps'])
print('  output_dir       :', config_data['training']['output_dir'])

### (Optional) Resume from a previous checkpoint

If the session was interrupted, uncomment the resume line and re-run Step 6.

In [ ]:
import glob, os

OUTPUT_DIR = config_data['training']['output_dir']
epoch_ckpts = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'checkpoint-epoch-*')))
print('Epoch checkpoints found:', epoch_ckpts or '(none)')

# None  -> train from scratch
# Uncomment the line below to resume from the latest saved epoch checkpoint:
# RESUME_CHECKPOINT = epoch_ckpts[-1] if epoch_ckpts else None
RESUME_CHECKPOINT = None
print('Resume from:', RESUME_CHECKPOINT)

## Step 6 — Build & Run Training

In [ ]:
train_config = {
    "config": cfg_file
}

# Build the CLI argument string (was missing in the original notebook)
train_args = train(train_config)
print('train_args:', train_args)

In [ ]:
# FIXED: correct path is /content/text-to-sql-finetune/... (not /text-to-sql-finetune/)
# FIXED: set PYTHONPATH so 'src.*' package imports work inside the subprocess
script = (
    'PYTHONPATH=/content/text-to-sql-finetune '
    'python /content/text-to-sql-finetune/src/training/train.py '
    + train_args
)
print('Command to run:')
print(script)

In [ ]:

%load_ext autoreload
%autoreload 2

In [ ]:
!{script}

## Step 7 — Verify Outputs

In [ ]:
import glob, os

OUTPUT_DIR = config_data['training']['output_dir']
print('Output dir:', OUTPUT_DIR, '\n')

epoch_ckpts = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'checkpoint-epoch-*')))
print('Epoch checkpoints:')
for c in epoch_ckpts:
    files = os.listdir(c)
    size  = sum(os.path.getsize(os.path.join(c, f))
                for f in files if os.path.isfile(os.path.join(c, f)))
    print('  %-35s  %.1f MB' % (os.path.basename(c), size / 1e6))

final = os.path.join(OUTPUT_DIR, 'final_lora')
if os.path.exists(final):
    print('\nFinal model:', os.listdir(final))
else:
    print('\nFinal model: (not saved yet)')

---
## Resume Guide — When Colab GPU Quota Expires

1. Re-run **Steps 1–5** (GPU check, install, Drive mount, source setup, config)
2. In the **Optional Resume** cell, uncomment:
   ```python
   RESUME_CHECKPOINT = epoch_ckpts[-1] if epoch_ckpts else None
   ```
3. Re-run **Step 6** — the trainer loads LoRA weights from Drive and continues

| `save_checkpoints` | Max progress lost |
|---|---|
| `1` | 1 epoch |
| `2` | 2 epochs |
| `None` | Everything (not recommended on Colab) |